# 🚀 TikTok 4.5B Viral Intelligence & Analytics Pipeline
### 289GB 허깅페이스 원본(`kuben-developer/tiktok-videos-4b`) 초고속 스트리밍 분석

> **💡 엔지니어링 핵심 포인트: 289GB를 내 컴퓨터에 다운로드받지 마세요!**  
> Parquet 포맷과 **DuckDB HTTPFS**를 활용하면, 수백 기가의 파일을 다운로드하지 않고도 허깅페이스 클라우드에 올려진 45억 건의 데이터에서 필요한 컬럼과 집계 데이터만 **수 초 만에 원격 쿼리**할 수 있습니다.

## 1. 분석 환경 및 DuckDB 엔진 세팅

In [ ]:
!pip install -q duckdb polars plotly huggingface_hub

import duckdb
import polars as pl
import plotly.express as px
import json

# DuckDB 인메모리 엔진 가동 및 원격 HTTPFS 확장 모듈 로드
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
print("✅ DuckDB HTTPFS 원격 쿼리 엔진 초기화 완료!")

## 2. 허깅페이스 289GB 데이터셋 구조 탐색 (무다운로드 메타데이터 스캔)
27개의 분할 Parquet 파일 중 1번 파일의 스키마와 데이터 샘플을 즉시 원격 스트리밍으로 읽어옵니다.

In [ ]:
# 허깅페이스 직접 다운로드 스트리밍 URL
parquet_url = "https://huggingface.co/datasets/kuben-developer/tiktok-videos-4b/resolve/main/data/train-00000-of-00027.parquet"

print("[*] 허깅페이스 원격 Parquet 스키마 조회 중...")
schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{parquet_url}')").fetchdf()
display(schema_df)

## 3. 실전 쿼리 1: 역대 최다 조회수(Play Count) Top 20 틱톡 영상 분석
어떤 캡션과 콘텐츠가 틱톡 전 세계 알고리즘을 지배했는지 확인합니다.

In [ ]:
query_top_views = f"""
SELECT 
    id AS video_id,
    title AS caption,
    play_count,
    digg_count AS likes,
    share_count,
    comment_count,
    round((digg_count * 100.0) / NULLIF(play_count, 0), 2) AS like_rate_pct,
    music_id,
    to_timestamp(create_time) AS created_at
FROM read_parquet('{parquet_url}')
WHERE play_count > 1000000
ORDER BY play_count DESC
LIMIT 20;
"""

top_views_df = con.execute(query_top_views).fetchdf()
display(top_views_df)

# 시각화
fig = px.bar(
    top_views_df,
    x='play_count',
    y='video_id',
    orientation='h',
    color='likes',
    hover_data=['caption', 'like_rate_pct', 'share_count'],
    title='🔥 틱톡 역대 최다 조회수 Top 20 영상 및 좋아요 분석'
)
fig.show()

## 4. 실전 쿼리 2: 바이럴을 주도한 최다 사용 BGM 음원(Music ID) 랭킹
틱톡 알고리즘 바이럴의 80%는 배경음악(BGM)에서 시작됩니다. 가장 많은 바이럴 영상을 만든 황금 음원을 추출합니다.

In [ ]:
query_top_music = f"""
SELECT 
    music_id,
    count(*) AS video_usage_count,
    sum(play_count) AS total_music_views,
    avg(digg_count) AS avg_likes_per_video
FROM read_parquet('{parquet_url}')
WHERE music_id IS NOT NULL AND music_id != ''
GROUP BY music_id
ORDER BY video_usage_count DESC
LIMIT 15;
"""

top_music_df = con.execute(query_top_music).fetchdf()
display(top_music_df)

fig_music = px.pie(
    top_music_df,
    values='total_music_views',
    names='music_id',
    title='🎵 틱톡 상위 바이럴 BGM 음원별 누적 조회수 점유율'
)
fig_music.show()

## 5. [선택 사항] 구글 드라이브로 영구 소장 백업
만약 구글 드라이브 용량(300GB 이상)이 충분하시다면, 아래 셀을 실행하여 허깅페이스 원본 데이터셋을 내 구글 드라이브에 직접 백그라운드 다운로드할 수 있습니다.

In [ ]:
# from google.colab import drive
# from huggingface_hub import snapshot_download

# drive.mount('/content/drive')
# snapshot_download(
#     repo_id='kuben-developer/tiktok-videos-4b',
#     repo_type='dataset',
#     local_dir='/content/drive/MyDrive/TikTok_4B_Dataset'
# )
# print('✅ 구글 드라이브 백업 완료!')